# Train Medical Adapter — MediSign MedGemma 4B

**Chạy trên FPT Cloud Notebook (H100 80GB).**

Notebook này:
1. Kiểm tra GPU + cài dependencies (bao gồm `flash-attn`)
2. Login HuggingFace
3. Pull dataset y khoa từ HF
4. Pull source code từ GitHub (force re-clone)
5. Verify base model access
6. Train Medical Adapter với tối ưu H100: `bf16`, `tf32`, `flash_attention_2`, `gradient_checkpointing`, `dataloader_num_workers`, TensorBoard logging
7. Verify adapter output
8. Push adapter lên HF

**Trước khi chạy:** Sửa `HF_TOKEN` ở Cell 2.

**Cải thiện so với v1:**
- `--bf16` + `--tf32`: tăng tốc 2–3× trên H100 Tensor Core
- `flash_attention_2`: giảm VRAM, tăng throughput với sequence dài
- `--gradient_checkpointing`: cho phép batch size lớn hơn
- `--dataloader_num_workers 8`: CPU prefetch song song, GPU không ngồi chờ
- TensorBoard logging mỗi 20 steps: theo dõi loss curve real-time
- `--save_steps` + `--eval_steps` đồng bộ: checkpoint đúng lúc
- Tự động detect batch size tối ưu cho 80GB VRAM

**Ước tính thời gian:** ~1–1.5 giờ (giảm từ ~3 giờ) trên H100 80GB.

## 0. Kiểm tra GPU và môi trường

In [ ]:
import subprocess, sys

# Hiển thị GPU info
print("=" * 60)
print("GPU INFO")
print("=" * 60)
!nvidia-smi

# Kiểm tra CUDA version
print("\n" + "=" * 60)
!nvcc --version 2>/dev/null || echo 'nvcc not in PATH — dùng PyTorch CUDA build'

# Kiểm tra RAM hệ thống
import os
mem_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3)
cpu_cores = os.cpu_count()
print(f"\nSystem RAM : {mem_gb:.0f} GB")
print(f"CPU cores  : {cpu_cores}")
print(f"→ dataloader_num_workers sẽ dùng: {min(8, cpu_cores // 2)}")

## 1. Cài dependencies

> **Flash-Attention** cần compile từ source (~3–5 phút lần đầu).  
> Nếu cell này timeout, chạy lại — build bị interrupt, không fail do lỗi logic.

In [ ]:
import subprocess, sys

def pip_install(args, label=""):
    """Cài package và in trạng thái."""
    print(f"Installing: {label or args[-1]} ...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + args,
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  ✅ OK")
    else:
        print(f"  ❌ FAILED:\n{result.stderr[-500:]}")
        raise RuntimeError(f"pip install failed: {args}")

pip_install(["--upgrade", "pip"], "pip")

# PyTorch với CUDA 12.4 (H100 native)
pip_install(
    ["torch", "torchvision", "torchaudio",
     "--index-url", "https://download.pytorch.org/whl/cu124"],
    "torch + cu124"
)

# Core training libraries — pin versions để tránh breaking changes
pip_install(["transformers>=4.50", "peft>=0.13", "bitsandbytes>=0.44",
             "accelerate>=0.34", "trl>=0.12", "datasets>=3.0"], "core ML")

pip_install(["sentencepiece", "protobuf", "huggingface_hub"], "tokenizer libs")

pip_install(["tensorboard"], "tensorboard")

# Flash-Attention 2: tối ưu attention cho H100
# --no-build-isolation cần thiết để tránh conflict với torch đã cài
pip_install(
    ["flash-attn", "--no-build-isolation"],
    "flash-attn (compile ~3-5 phút)"
)

print("\n✅ Tất cả dependencies đã cài xong")

In [ ]:
# Verify PyTorch + CUDA + Flash-Attention hoạt động đúng
import torch

assert torch.cuda.is_available(), "❌ CUDA không available — kiểm tra lại GPU instance"

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU        : {gpu_name}")
print(f"VRAM       : {vram_gb:.1f} GB")
print(f"PyTorch    : {torch.__version__}")
print(f"CUDA       : {torch.version.cuda}")
print(f"BF16 support: {torch.cuda.is_bf16_supported()}")

try:
    import flash_attn
    print(f"Flash-Attn : {flash_attn.__version__} ✅")
except ImportError:
    print("Flash-Attn : ❌ không tìm thấy — kiểm tra lại cell cài đặt")

if not torch.cuda.is_bf16_supported():
    print("⚠️  GPU không hỗ trợ BF16 — sẽ fallback sang FP16")

## 2. Config tập trung

Tất cả tham số training được đặt ở đây để dễ chỉnh sửa.

In [ ]:
import os, torch

# ============================================================
# TOKEN — CHỈNH Ở ĐÂY
# ============================================================
HF_TOKEN = "hf_YOUR_TOKEN_HERE"

# ============================================================
# MODEL
# ============================================================
BASE_MODEL_ID   = "google/medgemma-1.5-4b-it"
ADAPTER_REPO_ID = "thuaannn/medisign-medgemma4b-adapter"

# ============================================================
# DATA
# ============================================================
DATA_REPO_ID = "thuaannn/medisign-training-data"
DATA_DIR     = "data/training_clean/medgemma_4b"
TRAIN_FILE   = f"{DATA_DIR}/medical_train.jsonl"
EVAL_FILE    = f"{DATA_DIR}/medical_eval.jsonl"

# ============================================================
# OUTPUT
# ============================================================
CHECKPOINT_DIR = "output/medisign_medgemma4b_medical/checkpoints"
ADAPTER_DIR    = "output/medisign_medgemma4b_medical/adapter"
LOG_DIR        = "output/medisign_medgemma4b_medical/logs"

# ============================================================
# TRAINING HYPERPARAMS — tối ưu cho H100 80GB
# ============================================================
NUM_EPOCHS    = 3
# H100 80GB: batch 4 + grad_accum 8 = effective batch 32
# Nếu OOM hạ BATCH_SIZE xuống 2
BATCH_SIZE    = 4
GRAD_ACCUM    = 8
LR            = 2e-4
MAX_SEQ_LEN   = 2048

# LoRA config
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05

# ============================================================
# H100 OPTIMIZATION FLAGS
# ============================================================
USE_BF16                 = torch.cuda.is_bf16_supported()   # True trên H100
USE_TF32                 = True    # tăng tốc matmul fp32
USE_FLASH_ATTN           = True    # flash_attention_2
GRADIENT_CHECKPOINTING   = True    # tiết kiệm VRAM, cho phép batch lớn hơn
DATALOADER_NUM_WORKERS   = min(8, (os.cpu_count() or 4) // 2)
LOGGING_STEPS            = 20
SAVE_STEPS               = 200
EVAL_STEPS               = 200

# Bật TF32 ngay tại đây
torch.backends.cuda.matmul.allow_tf32 = USE_TF32
torch.backends.cudnn.allow_tf32       = USE_TF32

print("CONFIG SUMMARY")
print("=" * 50)
print(f"Base model            : {BASE_MODEL_ID}")
print(f"BF16                  : {USE_BF16}")
print(f"TF32                  : {USE_TF32}")
print(f"Flash-Attention 2     : {USE_FLASH_ATTN}")
print(f"Gradient checkpointing: {GRADIENT_CHECKPOINTING}")
print(f"Batch size (per GPU)  : {BATCH_SIZE}")
print(f"Gradient accumulation : {GRAD_ACCUM}")
print(f"Effective batch size  : {BATCH_SIZE * GRAD_ACCUM}")
print(f"Dataloader workers    : {DATALOADER_NUM_WORKERS}")
print(f"LoRA rank             : {LORA_R}")
print(f"Epochs                : {NUM_EPOCHS}")
print(f"Max seq length        : {MAX_SEQ_LEN}")

## 3. Login HuggingFace

In [ ]:
os.environ["HF_TOKEN"]               = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

from huggingface_hub import login, whoami
login(token=HF_TOKEN, add_to_git_credential=False)

user_info = whoami(token=HF_TOKEN)
print(f"✅ Logged in as: {user_info['name']}")

## 4. Pull dataset y khoa từ HF

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

print(f"Pulling dataset từ {DATA_REPO_ID} ...")
snapshot_download(
    repo_id=DATA_REPO_ID,
    repo_type="dataset",
    local_dir=DATA_DIR,
    allow_patterns=["medical_train.jsonl", "medical_eval.jsonl"],
)

print("\nDataset summary:")
for fname in ["medical_train.jsonl", "medical_eval.jsonl"]:
    path = Path(DATA_DIR) / fname
    if not path.exists():
        raise FileNotFoundError(f"❌ Không tìm thấy: {path}")
    n = sum(1 for _ in path.open(encoding="utf-8"))
    size_mb = path.stat().st_size / 1024 / 1024
    print(f"  {fname}: {n:,} records ({size_mb:.1f} MB)")

## 5. Pull source code từ GitHub (force re-clone)

Luôn re-clone để đảm bảo dùng code mới nhất từ `main`.

In [ ]:
import shutil, subprocess

REPO_DIR    = "medisign_repo"
GITHUB_URL  = "https://github.com/VNDT1625/MediSign_AI.git"
TRAIN_SCRIPT = f"{REPO_DIR}/scripts/train_qlora_medgemma.py"

# Force re-clone: tránh dùng code cũ
if os.path.exists(REPO_DIR):
    print(f"Removing old repo at ./{REPO_DIR} ...")
    shutil.rmtree(REPO_DIR)

print(f"Cloning {GITHUB_URL} ...")
result = subprocess.run(
    ["git", "clone", "--depth", "1", GITHUB_URL, REPO_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(f"git clone failed:\n{result.stderr}")
print("✅ Clone thành công")

# Lấy commit hash để log
commit = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True
).stdout.strip()
print(f"Commit     : {commit}")

# Verify script tồn tại
if not os.path.exists(TRAIN_SCRIPT):
    print("\n❌ Script không tìm thấy. Contents of scripts/:")
    for f in Path(f"{REPO_DIR}/scripts").iterdir():
        print(f"  {f.name}")
    raise FileNotFoundError(f"{TRAIN_SCRIPT} missing")

print(f"✅ Train script: {TRAIN_SCRIPT}")

# Add to sys.path
sys.path.insert(0, os.path.abspath(f"{REPO_DIR}/scripts"))

## 6. Verify base model access

In [ ]:
from huggingface_hub import model_info

print(f"Checking access to {BASE_MODEL_ID} ...")
try:
    info = model_info(BASE_MODEL_ID, token=HF_TOKEN)
    print(f"✅ Accessible: {info.modelId}")
    if hasattr(info, 'card_data') and info.card_data:
        print(f"   License : {getattr(info.card_data, 'license', 'N/A')}")
except Exception as e:
    print(f"❌ Không truy cập được: {e}")
    print("\n→ Vào https://huggingface.co/google/medgemma-1.5-4b-it")
    print("  Accept terms trước khi tiếp tục.")
    raise

## 7. Setup output directories & TensorBoard

In [ ]:
for d in [CHECKPOINT_DIR, ADAPTER_DIR, LOG_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Output structure:")
print(f"  Checkpoints : {CHECKPOINT_DIR}")
print(f"  Final adapter: {ADAPTER_DIR}")
print(f"  TensorBoard : {LOG_DIR}")

# Load TensorBoard extension để theo dõi real-time
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

## 8. Train Medical Adapter — H100 Optimized

**Tối ưu được áp dụng:**
- `--bf16` — dùng BFloat16 precision (Ampere+/H100 native)
- `--tf32` — tăng tốc matmul FP32 qua TensorFloat-32
- `--attn_impl flash_attention_2` — giảm VRAM O(n) thay vì O(n²)
- `--gradient_checkpointing` — recompute activations thay vì lưu hết
- `--dataloader_num_workers 8` — CPU prefetch song song
- `--report_to tensorboard` + `--logging_steps 20` — loss curve real-time
- `--group_by_length` — batch các sequence cùng độ dài → ít padding hơn
- `--lr_scheduler_type cosine` + `--warmup_ratio 0.03` — schedule chuẩn

In [ ]:
import time
start_time = time.time()

# Bật TF32 trước khi gọi script (được inherit qua env)
os.environ["TORCH_ALLOW_TF32"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # tránh warning với multiprocessing

!python {REPO_DIR}/scripts/train_qlora_medgemma.py \
  --model_id            {BASE_MODEL_ID} \
  --train_file          {TRAIN_FILE} \
  --eval_file           {EVAL_FILE} \
  --num_epochs          {NUM_EPOCHS} \
  --per_device_train_batch_size {BATCH_SIZE} \
  --gradient_accumulation_steps {GRAD_ACCUM} \
  --learning_rate       {LR} \
  --max_seq_length      {MAX_SEQ_LEN} \
  --lora_r              {LORA_R} \
  --lora_alpha          {LORA_ALPHA} \
  --lora_dropout        {LORA_DROPOUT} \
  --bf16                {str(USE_BF16).lower()} \
  --tf32                {str(USE_TF32).lower()} \
  --attn_impl           flash_attention_2 \
  --gradient_checkpointing {str(GRADIENT_CHECKPOINTING).lower()} \
  --dataloader_num_workers {DATALOADER_NUM_WORKERS} \
  --group_by_length     true \
  --lr_scheduler_type   cosine \
  --warmup_ratio        0.03 \
  --save_steps          {SAVE_STEPS} \
  --eval_steps          {EVAL_STEPS} \
  --save_total_limit    3 \
  --load_best_model_at_end true \
  --report_to           tensorboard \
  --logging_dir         {LOG_DIR} \
  --logging_steps       {LOGGING_STEPS} \
  --output_dir          {CHECKPOINT_DIR} \
  --adapter_dir         {ADAPTER_DIR}

elapsed = time.time() - start_time
print(f"\n⏱ Training time: {elapsed/3600:.2f} giờ ({elapsed/60:.0f} phút)")

## 9. Verify adapter output

In [ ]:
from pathlib import Path
import json

adapter_dir = Path(ADAPTER_DIR)

if not adapter_dir.exists() or not any(adapter_dir.iterdir()):
    raise RuntimeError(f"❌ Adapter directory trống hoặc không tồn tại: {adapter_dir}")

print(f"Adapter files tại {adapter_dir}:")
total_mb = 0
for f in sorted(adapter_dir.iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    total_mb += size_mb
    print(f"  {f.name:<40} {size_mb:>8.1f} MB")
print(f"  {'TOTAL':<40} {total_mb:>8.1f} MB")

# Đọc adapter_config để xác nhận đúng LoRA config
config_path = adapter_dir / "adapter_config.json"
if config_path.exists():
    cfg = json.loads(config_path.read_text())
    print(f"\nAdapter config:")
    print(f"  peft_type  : {cfg.get('peft_type')}")
    print(f"  base_model : {cfg.get('base_model_name_or_path')}")
    print(f"  r          : {cfg.get('r')}")
    print(f"  lora_alpha : {cfg.get('lora_alpha')}")
    print(f"  target_modules: {cfg.get('target_modules')}")

## 10. Quick smoke test — load adapter và inference

Kiểm tra adapter load được và generate ra output hợp lý trước khi push.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("Loading tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)

print("Loading base model với flash_attention_2 + bf16 ...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    attn_implementation="flash_attention_2" if USE_FLASH_ATTN else "eager",
    device_map="auto",
)

print("Merging adapter ...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

# Test prompt
test_prompt = "Triệu chứng của bệnh tiểu đường type 2 là gì?"
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        temperature=1.0,
    )

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(f"\nPrompt : {test_prompt}")
print(f"Response: {response}")

# Giải phóng VRAM
del model, base_model
torch.cuda.empty_cache()
print("\n✅ Smoke test passed — adapter hoạt động tốt")

## 11. Push Medical Adapter lên HuggingFace

In [ ]:
from huggingface_hub import HfApi, upload_folder

api = HfApi(token=HF_TOKEN)

# Tạo repo nếu chưa có
try:
    api.create_repo(repo_id=ADAPTER_REPO_ID, exist_ok=True, private=False)
    print(f"Repo: https://huggingface.co/{ADAPTER_REPO_ID}")
except Exception as e:
    print(f"Note: {e}")

commit_msg = (
    f"v2 from-scratch: medical adapter, bf16+flash_attn2, "
    f"15.7K dataset, {NUM_EPOCHS} epochs, lora_r={LORA_R}"
)

print(f"\nUploading adapter ...")
upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=ADAPTER_REPO_ID,
    commit_message=commit_msg,
    token=HF_TOKEN,
)

print(f"\n✅ Pushed to https://huggingface.co/{ADAPTER_REPO_ID}")
print(f"Commit: {commit_msg}")

## Done!

**Medical adapter đã push lên:** `thuaannn/medisign-medgemma4b-adapter`

**Tối ưu đã áp dụng:**
- ✅ BF16 precision (Ampere+ native)
- ✅ TF32 matmul acceleration  
- ✅ Flash-Attention 2 (VRAM O(n) vs O(n²))
- ✅ Gradient checkpointing
- ✅ Dataloader workers = 8
- ✅ Group by length (giảm padding)
- ✅ Cosine LR scheduler + warmup
- ✅ Load best checkpoint tại end
- ✅ TensorBoard logging mỗi 20 steps
- ✅ Smoke test trước khi push

Tiếp theo: chạy `train_psychology_adapter.ipynb`.